In [7]:
import numpy as np

# --- 1. USER INPUTS ---
print("=== EV Charging Station Placement via GWO ===\n")
try:
    M = int(input("Enter the number of charging stations to place (default 3): ") or "3")
    N = int(input("Enter the population size of wolves (default 15): ") or "15")
    MaxIter = int(input("Enter the maximum GWO iterations (default 100): ") or "100")
except ValueError:
    print("Invalid numeric input! Falling back to defaults.")
    M, N, MaxIter = 3, 15, 100

# Default demand points: [X, Y, Traffic_Weight]
default_demand = np.array([
    [15, 20, 50],
    [30, 85, 30],
    [80, 75, 90],
    [45, 50, 40],
    [90, 10, 70],
    [10, 90, 20],
    [60, 30, 60],
    [25, 40, 45],
    [75, 85, 80],
    [50, 10, 35]
])

print("\nDo you want to enter custom demand points?")
custom_choice = input("Type 'yes' to enter manually, or press Enter to use default demand coordinates: ").strip().lower()

demand_points = []
if custom_choice == 'yes':
    try:
        num_points = int(input("How many demand points do you want to add? "))
        for k in range(num_points):
            coords_str = input(f"Enter X, Y, and Traffic Weight for point {k+1} (comma-separated, e.g., 25,45,60): ")
            x_pt, y_pt, wt = map(float, coords_str.split(","))
            demand_points.append([x_pt, y_pt, wt])
        demand_points = np.array(demand_points)
    except Exception as e:
        print(f"Error reading input: {e}. Falling back to default demand points.")
        demand_points = default_demand
else:
    demand_points = default_demand

# --- 2. OPTIMIZER ENGINE ---
X_min, X_max = 0, 100
Y_min, Y_max = 0, 100
wolves = np.random.uniform(low=0, high=100, size=(N, M * 2))

def calculate_cost(wolf_pos, demand):
    stations = wolf_pos.reshape((M, 2))
    total_travel_cost = 0
    for dp in demand:
        dp_coord = dp[:2]
        traffic_weight = dp[2]
        distances = np.sqrt(np.sum((stations - dp_coord)**2, axis=1))
        total_travel_cost += np.min(distances) * traffic_weight
    return total_travel_cost

# Initial Hierarchy Setup
fitness = np.zeros(N)
for i in range(N):
    fitness[i] = calculate_cost(wolves[i], demand_points)

sorted_indices = np.argsort(fitness)
X_alpha = wolves[sorted_indices[0]].copy()
X_beta  = wolves[sorted_indices[1]].copy()
X_delta = wolves[sorted_indices[2]].copy()

# GWO Loop
for t in range(MaxIter):
    a = 2 - (2 * t / MaxIter)
    for i in range(N):
        for j in range(M * 2):
            r1, r2 = np.random.rand(), np.random.rand()
            A1, C1 = 2 * a * r1 - a, 2 * r2
            D_alpha = abs(C1 * X_alpha[j] - wolves[i, j])
            X1 = X_alpha[j] - A1 * D_alpha

            r1, r2 = np.random.rand(), np.random.rand()
            A2, C2 = 2 * a * r1 - a, 2 * r2
            D_beta = abs(C2 * X_beta[j] - wolves[i, j])
            X2 = X_beta[j] - A2 * D_beta

            r1, r2 = np.random.rand(), np.random.rand()
            A3, C3 = 2 * a * r1 - a, 2 * r2
            D_delta = abs(C3 * X_delta[j] - wolves[i, j])
            X3 = X_delta[j] - A3 * D_delta

            wolves[i, j] = (X1 + X2 + X3) / 3

            if j % 2 == 0:
                wolves[i, j] = np.clip(wolves[i, j], X_min, X_max)
            else:
                wolves[i, j] = np.clip(wolves[i, j], Y_min, Y_max)

    for i in range(N):
        fitness[i] = calculate_cost(wolves[i], demand_points)

    sorted_indices = np.argsort(fitness)
    X_alpha = wolves[sorted_indices[0]].copy()
    X_beta  = wolves[sorted_indices[1]].copy()
    X_delta = wolves[sorted_indices[2]].copy()

# --- 3. OUTPUT OPTIMAL LAYOUT ---
final_stations = X_alpha.reshape((M, 2))
print("\n" + "="*45)
print(f"OPTIMAL CHARGING STATION LOCATIONS (Fitness: {fitness[sorted_indices[0]]:.2f})")
print("="*45)
for index, station in enumerate(final_stations):
    print(f"Station {index+1:02d} -> X: {station[0]:.4f}, Y: {station[1]:.4f}")
print("="*45)


=== EV Charging Station Placement via GWO ===

Enter the number of charging stations to place (default 3): 2
Enter the population size of wolves (default 15): 15
Enter the maximum GWO iterations (default 100): 100

Do you want to enter custom demand points?
Type 'yes' to enter manually, or press Enter to use default demand coordinates: yes
How many demand points do you want to add? 5
Enter X, Y, and Traffic Weight for point 1 (comma-separated, e.g., 25,45,60): 10, 10, 100
Enter X, Y, and Traffic Weight for point 2 (comma-separated, e.g., 25,45,60): 90, 90, 90
Enter X, Y, and Traffic Weight for point 3 (comma-separated, e.g., 25,45,60): 15, 80, 20
Enter X, Y, and Traffic Weight for point 4 (comma-separated, e.g., 25,45,60): 85, 15, 30
Enter X, Y, and Traffic Weight for point 5 (comma-separated, e.g., 25,45,60): 50, 50, 50

OPTIMAL CHARGING STATION LOCATIONS (Fitness: 6509.59)
Station 01 -> X: 90.0811, Y: 89.9218
Station 02 -> X: 9.8968, Y: 10.1809
